In [1]:
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

In [2]:
SOURCE_DATASET = Path(r"C:\Users\sujan\Desktop\final project2\DATASET\RiceDiseaseDataset")

OUTPUT_DATASET = Path(r"C:\Users\sujan\Desktop\final project2\DATASET\Processed_RiceDisease_Dataset")

TRAIN_DIR = OUTPUT_DATASET / "train"
VAL_DIR = OUTPUT_DATASET / "val"
TEST_DIR = OUTPUT_DATASET / "test"

In [3]:
for folder in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [4]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")

classes = sorted(
    [
        folder.name
        for folder in SOURCE_DATASET.iterdir()
        if folder.is_dir()
    ]
)

for cls in classes:

    class_path = SOURCE_DATASET / cls

    images = [
        img
        for img in class_path.iterdir()
        if img.suffix.lower() in IMAGE_EXTENSIONS
    ]

    # 70% Train
    train_images, temp_images = train_test_split(
        images,
        test_size=0.30,
        random_state=42,
        shuffle=True
    )

    # Remaining 30% -> 15% Validation + 15% Test
    val_images, test_images = train_test_split(
        temp_images,
        test_size=0.50,
        random_state=42,
        shuffle=True
    )

    # Create class folders
    (TRAIN_DIR / cls).mkdir(exist_ok=True)
    (VAL_DIR / cls).mkdir(exist_ok=True)
    (TEST_DIR / cls).mkdir(exist_ok=True)

    # Copy images
    for img in train_images:
        shutil.copy2(img, TRAIN_DIR / cls / img.name)

    for img in val_images:
        shutil.copy2(img, VAL_DIR / cls / img.name)

    for img in test_images:
        shutil.copy2(img, TEST_DIR / cls / img.name)

print("Dataset split completed successfully.")

Dataset split completed successfully.


In [5]:
print("=" * 70)
print(f"{'Class':30} {'Train':>8} {'Val':>8} {'Test':>8}")
print("=" * 70)

for cls in classes:

    train_count = len(list((TRAIN_DIR / cls).glob("*")))
    val_count = len(list((VAL_DIR / cls).glob("*")))
    test_count = len(list((TEST_DIR / cls).glob("*")))

    print(f"{cls:30} {train_count:8} {val_count:8} {test_count:8}")

print("=" * 70)

Class                             Train      Val     Test
bacterial_leaf_blight               253       54       55
brown_spot                          274       59       59
healthy                             305       65       66
leaf_blast                          294       63       63
leaf_scald                          249       54       54
leaf_smut                            28        6        6
narrow_brown_spot                   286       61       62


In [ ]:
#Data Augmentation for the leaf_smut
from pathlib import Path
from PIL import Image
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [7]:
TRAIN_DIR = Path(
    r"C:\Users\sujan\Desktop\final project2\DATASET\Processed_RiceDisease_Dataset\train"
)

LEAF_SMUT_DIR = TRAIN_DIR / "leaf_smut"

print("Leaf Smut folder:")
print(LEAF_SMUT_DIR)

Leaf Smut folder:
C:\Users\sujan\Desktop\final project2\DATASET\Processed_RiceDisease_Dataset\train\leaf_smut


In [8]:
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp"
}

leaf_smut_images = [
    file for file in LEAF_SMUT_DIR.iterdir()
    if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS
]

print("Current leaf_smut images:", len(leaf_smut_images))

Current leaf_smut images: 28


In [9]:
class_counts = {}

for class_dir in TRAIN_DIR.iterdir():

    if class_dir.is_dir():

        count = sum(
            1 for file in class_dir.iterdir()
            if file.is_file()
            and file.suffix.lower() in IMAGE_EXTENSIONS
        )

        class_counts[class_dir.name] = count

print(class_counts)

TARGET_COUNT = max(class_counts.values())

print("\nTarget training count:", TARGET_COUNT)

{'bacterial_leaf_blight': 253, 'brown_spot': 274, 'healthy': 305, 'leaf_blast': 294, 'leaf_scald': 249, 'leaf_smut': 28, 'narrow_brown_spot': 286}

Target training count: 305


In [10]:
datagen = ImageDataGenerator(
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.20,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode="nearest"
)

In [11]:
current_count = len(leaf_smut_images)
images_needed = TARGET_COUNT - current_count

print("Current images :", current_count)
print("Target images  :", TARGET_COUNT)
print("Images needed  :", images_needed)

Current images : 28
Target images  : 305
Images needed  : 277


In [12]:
generated = 0

while generated < images_needed:

    for image_path in leaf_smut_images:

        if generated >= images_needed:
            break

        try:
            image = Image.open(image_path).convert("RGB")

            # Add batch dimension
            image_array = __import__("numpy").array(image)
            image_array = image_array.reshape(
                (1,) + image_array.shape
            )

            iterator = datagen.flow(
                image_array,
                batch_size=1
            )

            augmented_image = next(iterator)[0].astype("uint8")

            output_name = (
                f"aug_leaf_smut_{generated + 1:04d}.jpg"
            )

            output_path = LEAF_SMUT_DIR / output_name

            Image.fromarray(augmented_image).save(
                output_path,
                quality=95
            )

            generated += 1

        except Exception as e:
            print(f"Error processing {image_path.name}: {e}")

print(f"\nGenerated {generated} augmented images.")


Generated 277 augmented images.


In [13]:
final_count = sum(
    1 for file in LEAF_SMUT_DIR.iterdir()
    if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS
)

print("Final leaf_smut training images:", final_count)

Final leaf_smut training images: 305


In [14]:
from pathlib import Path

DATASET_PATH = Path(
    r"C:\Users\sujan\Desktop\final project2\DATASET\Processed_RiceDisease_Dataset"
)

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp"
}

splits = ["train", "val", "test"]

grand_total = 0

print("=" * 80)
print("FINAL PROCESSED DATASET SUMMARY")
print("=" * 80)

for split in splits:

    split_path = DATASET_PATH / split
    split_total = 0

    print(f"\n{'=' * 80}")
    print(f"{split.upper()}")
    print(f"{'=' * 80}")
    print(f"{'Class':30} {'Images':>10}")
    print("-" * 45)

    for class_dir in sorted(split_path.iterdir()):

        if not class_dir.is_dir():
            continue

        count = sum(
            1 for file in class_dir.iterdir()
            if file.is_file()
            and file.suffix.lower() in IMAGE_EXTENSIONS
        )

        split_total += count

        print(f"{class_dir.name:30} {count:10}")

    print("-" * 45)
    print(f"{'TOTAL ' + split.upper():30} {split_total:10}")

    grand_total += split_total

print("\n" + "=" * 80)
print(f"{'GRAND TOTAL':30} {grand_total:10}")
print("=" * 80)

FINAL PROCESSED DATASET SUMMARY

TRAIN
Class                              Images
---------------------------------------------
bacterial_leaf_blight                 253
brown_spot                            274
healthy                               305
leaf_blast                            294
leaf_scald                            249
leaf_smut                             305
narrow_brown_spot                     286
---------------------------------------------
TOTAL TRAIN                          1966

VAL
Class                              Images
---------------------------------------------
bacterial_leaf_blight                  54
brown_spot                             59
healthy                                65
leaf_blast                             63
leaf_scald                             54
leaf_smut                               6
narrow_brown_spot                      61
---------------------------------------------
TOTAL VAL                             362

TEST
Class     

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [1]:
from pathlib import Path
import os

LEAF_SMUT_DIR = Path(
    r"C:\Users\sujan\Desktop\final project2\DATASET\Processed_RiceDisease_Dataset\train\leaf_smut"
)

# Get all augmented images
augmented_images = sorted(LEAF_SMUT_DIR.glob("aug_leaf_smut_*.jpg"))

print("Current augmented images:", len(augmented_images))

# Keep only the first 122 augmented images
images_to_delete = augmented_images[122:]

for img in images_to_delete:
    os.remove(img)

print(f"Deleted {len(images_to_delete)} augmented images.")
print("Remaining augmented images:", len(list(LEAF_SMUT_DIR.glob('aug_leaf_smut_*.jpg'))))

Current augmented images: 277
Deleted 155 augmented images.
Remaining augmented images: 122


In [2]:
from pathlib import Path

LEAF_SMUT_DIR = Path(
    r"C:\Users\sujan\Desktop\final project2\DATASET\Processed_RiceDisease_Dataset\train\leaf_smut"
)

print("Final leaf_smut images:", len(list(LEAF_SMUT_DIR.glob("*.*"))))

Final leaf_smut images: 150


In [1]:
import os

# Dataset path
dataset_path = r"C:\Users\sujan\Desktop\final project2\DATASET\Processed_RiceDisease_Dataset"

splits = ["train", "val", "test"]

total_all = 0

print("=" * 65)
print("RICE DISEASE DATASET IMAGE COUNT")
print("=" * 65)

for split in splits:

    split_path = os.path.join(dataset_path, split)

    split_total = 0

    print(f"\n{split.upper()}")
    print("-" * 65)

    for class_name in sorted(os.listdir(split_path)):

        class_path = os.path.join(split_path, class_name)

        if os.path.isdir(class_path):

            image_count = 0

            for file in os.listdir(class_path):
                if file.lower().endswith((
                    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
                )):
                    image_count += 1

            print(f"{class_name:<25}: {image_count}")

            split_total += image_count

    print("-" * 65)
    print(f"{split.capitalize():<25} Total: {split_total}")

    total_all += split_total


print("\n" + "=" * 65)
print(f"TOTAL IMAGES (Train + Val + Test): {total_all}")
print("=" * 65)

RICE DISEASE DATASET IMAGE COUNT

TRAIN
-----------------------------------------------------------------
bacterial_leaf_blight    : 253
brown_spot               : 274
healthy                  : 305
leaf_blast               : 294
leaf_scald               : 249
leaf_smut                : 150
narrow_brown_spot        : 286
-----------------------------------------------------------------
Train                     Total: 1811

VAL
-----------------------------------------------------------------
bacterial_leaf_blight    : 54
brown_spot               : 59
healthy                  : 65
leaf_blast               : 63
leaf_scald               : 54
leaf_smut                : 6
narrow_brown_spot        : 61
-----------------------------------------------------------------
Val                       Total: 362

TEST
-----------------------------------------------------------------
bacterial_leaf_blight    : 55
brown_spot               : 59
healthy                  : 66
leaf_blast               : 6